# GPT, from scratch

This is the *Practice* step of `unit_07_gpt.md`. Do the Cold Attempt there first.

Work top to bottom. Each milestone is one cell of stubs followed by a grader cell.
The grader stops at your first failure so there is always exactly one thing in front of you.

**Rules of engagement**
- Don't open the lecture. Don't open nanoGPT or the lecture's repo.
- Stuck on an *idea* for 20 min → ask the coaching chat for a hint.
- Stuck on *PyTorch syntax* → ask immediately, zero learning value in that.
- **Before you run a grader cell, say out loud what you expect to happen.**

By this lecture PyTorch itself is scaffolding. Data loading, the tokenizer, batching, the
training loop and the sampling plumbing are given below and are *not* the lesson. The lesson
is how a token gathers information from the tokens before it, and only those.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from test_gpt import grade

torch.manual_seed(1337)

## Given: data, tokenizer, batches, evaluation, training loop

Read it once so you know the names. Nothing in this cell is the idea.

- `vocab_size` characters, `encode` / `decode` at the character level.
- `get_batch(split, batch_size, block_size)` returns `x, y` of shape `(B, T)`, where `y` is `x` shifted one to the right.
- `estimate_loss(model, ...)` averages the loss over a few batches of train and val.
- `train(model, steps, lr, ...)` is the whole training loop.

In [ ]:
with open('../data/tinyshakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]


def get_batch(split, batch_size, block_size):
    """Random (B, T) chunks x and their one-step-ahead targets y."""
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x, y


@torch.no_grad()
def estimate_loss(model, batch_size, block_size, eval_iters=50):
    model.eval()
    out = {}
    for split in ('train', 'val'):
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split, batch_size, block_size)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


def train(model, steps, lr, batch_size=32, block_size=8, eval_every=200):
    """Plain AdamW loop. Prints train/val loss every eval_every steps."""
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for step in range(steps):
        if step % eval_every == 0 or step == steps - 1:
            l = estimate_loss(model, batch_size, block_size)
            print(f"step {step:5d}  train {l['train']:.4f}  val {l['val']:.4f}")
        xb, yb = get_batch('train', batch_size, block_size)
        _, loss = model(xb, yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    return model


def sample_from_logits(logits):
    """Given logits of shape (B, vocab), return one sampled token id per row, shape (B, 1)."""
    probs = F.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)


print(f"{len(text):,} characters, vocab {vocab_size}")
xb, yb = get_batch('train', 4, 8)
print(xb.shape, yb.shape)

## Milestone 1 — the bigram language model, as an `nn.Module`

The simplest possible model: the next-token logits are a lookup of the current token. Two
things to get right that carry through to the GPT unchanged: how a `(B, T, C)` tensor of
logits meets a `(B, T)` tensor of targets in the loss, and how `generate` turns a model that
scores one step into a loop that writes text.

In [ ]:
class BigramLanguageModel(nn.Module):
    """Next-token logits are a table lookup on the current token.

    Contract (shared with GPTLanguageModel later):
      forward(idx, targets=None) -> (logits, loss)
        idx      (B, T) long tensor of token ids
        targets  (B, T) long tensor or None
        logits   (B, T, vocab_size) float tensor
        loss     scalar tensor (mean cross entropy over every (b, t)) or None when targets is None
      generate(idx, max_new_tokens) -> (B, T + max_new_tokens) long tensor
        idx is the context; the returned tensor starts with it.
        Before each forward call, crop the context to its last self.block_size tokens.
        Use sample_from_logits (given above) to draw the next id from the LAST time step.
    """

    def __init__(self, vocab_size, block_size):
        super().__init__()
        self.block_size = block_size
        raise NotImplementedError

    def forward(self, idx, targets=None):
        raise NotImplementedError

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        raise NotImplementedError

In [ ]:
grade(BigramLanguageModel=BigramLanguageModel, upto=1)

In [ ]:
# once milestone 1 passes: train it, look at the loss, look at the text (~10 s)
bigram = BigramLanguageModel(vocab_size, block_size=8)
train(bigram, steps=2000, lr=1e-2, batch_size=32, block_size=8, eval_every=500)
print(decode(bigram.generate(torch.zeros((1, 1), dtype=torch.long), 300)[0].tolist()))

## Milestone 2 — the trick: averaging the past, three ways

Before any attention: give every token the *mean* of itself and all the tokens before it.
Write it three times. All three must produce the same numbers. The third one is the shape
attention will take.

In [ ]:
def agg_loop(x):
    """x: (B, T, C). Return out of the same shape where out[b, t] is the mean of x[b, 0..t] inclusive.
    Do it the slow way: explicit loops over b and t."""
    raise NotImplementedError


def agg_tril(x):
    """Same result as agg_loop, but as a single matrix multiply with a (T, T) weight matrix
    built from torch.tril. Build the matrix from x.shape, not a hard-coded T."""
    raise NotImplementedError


def agg_softmax(x):
    """Same result again. This time start from a (T, T) matrix of zeros, set the entries that
    must not contribute to -inf, and take a softmax over the right axis to get the weights."""
    raise NotImplementedError

In [ ]:
grade(BigramLanguageModel=BigramLanguageModel,
      agg_loop=agg_loop, agg_tril=agg_tril, agg_softmax=agg_softmax, upto=2)

## Milestone 3 — one self-attention head. The crux.

The averaging above uses fixed, uniform weights. Attention makes the weights *data dependent*.
Same triangle, same softmax; the numbers before the mask now come from the data.

**Head(n_embd, head_size, block_size, dropout=0.0)**
- Holds: `self.key`, `self.query`, `self.value`, each `nn.Linear(n_embd, head_size, bias=False)`
  (weight shape `(head_size, n_embd)`); a `(block_size, block_size)` lower-triangular mask
  registered as a buffer, not a parameter.
- `forward(x)` for `x` of shape `(B, T, n_embd)`, `T <= block_size`:
  `softmax(mask(q @ kᵀ · head_size^-0.5)) @ v`, where `k, q, v` are the three Linears applied
  to `x`, masked entries (key position > query position) are `-inf` before the softmax, and
  the softmax runs over the key axis.
- Returns `(B, T, head_size)`. Output position `t` depends only on `x[:, :t+1]`. Must work
  for `T < block_size` (`generate` calls it that way). Gradient must reach all three Linears.

The grader checks, in order: the three Linears exist with the right shapes and no bias →
output shape → values against a reference built from your own weights (it names scaling,
mask, softmax axis, or a k/q swap if your output matches one of those) → no future leak →
the past is actually used → `T < block_size` → gradients reach key/query/value.

Predict before writing: with `q` and `k` of unit variance, what is the variance of one entry
of `q @ k.T`? The cell after the stub lets you measure it.

In [ ]:
class Head(nn.Module):
    """One head of causal self-attention.

    __init__(n_embd, head_size, block_size, dropout=0.0)
      Create self.key, self.query, self.value as nn.Linear(n_embd, head_size, bias=False).
      Keep the (block_size, block_size) lower-triangular mask as a buffer (register_buffer),
      not a parameter.
    forward(x) -> (B, T, head_size)
      x: (B, T, n_embd). T may be smaller than block_size.
      Tokens must only aggregate from positions <= their own.
    """

    def __init__(self, n_embd, head_size, block_size, dropout=0.0):
        super().__init__()
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError

In [ ]:
# measure, don't guess: variance of raw scores vs scaled scores, for unit-variance q and k
B, T, hs = 4, 8, 16
k = torch.randn(B, T, hs)
q = torch.randn(B, T, hs)
raw = q @ k.transpose(-2, -1)
print('var(k)', k.var().item(), ' var(q)', q.var().item())
print('var(q @ k^T)          ', raw.var().item())
print('var(q @ k^T / sqrt(hs))', (raw * hs ** -0.5).var().item())
print('softmax of a row of raw scores   ', F.softmax(raw[0, -1], dim=-1).max().item())
print('softmax of a row of scaled scores', F.softmax(raw[0, -1] * hs ** -0.5, dim=-1).max().item())

In [ ]:
grade(BigramLanguageModel=BigramLanguageModel,
      agg_loop=agg_loop, agg_tril=agg_tril, agg_softmax=agg_softmax,
      Head=Head, upto=3)

## Milestone 4 — many heads, and a place to think

Attention is the *communication* step: tokens exchange information. It contains no
nonlinearity of its own. The feed-forward network is the *computation* step, applied to
every token by itself.

**MultiHeadAttention(n_embd, num_heads, head_size, block_size, dropout=0.0)**
- Holds: `self.heads`, an `nn.ModuleList` of `num_heads` × `Head(n_embd, head_size, block_size, dropout)`;
  `self.proj = nn.Linear(num_heads * head_size, n_embd)` (weight shape `(n_embd, num_heads * head_size)`).
- `forward(x)`: `proj(cat([h(x) for h in heads], dim=-1))`.
- Returns `(B, T, n_embd)`. Still causal.

**FeedForward(n_embd, dropout=0.0)**
- Holds: `Linear(n_embd, 4*n_embd)` → `ReLU` → `Linear(4*n_embd, n_embd)`, both Linears with
  bias. For `n_embd = 32` that is 8352 parameters.
- `forward(x)`: that chain, applied along the last axis.
- Returns the same shape as `x`. Position `t`'s output depends only on position `t`'s input.

The grader checks, in order: `heads` is a ModuleList of `num_heads` → `proj` weight shape →
MHA output shape → values equal `proj(concat of the heads over the channel axis)` (it says so
if the proj is missing or the concat axis is wrong) → no future leak → FeedForward output
shape → parameter count → not affine → per-token.

In [ ]:
class MultiHeadAttention(nn.Module):
    """num_heads Heads run in parallel, results concatenated along the channel axis, then
    projected back to n_embd.

    __init__(n_embd, num_heads, head_size, block_size, dropout=0.0)
      self.heads: an nn.ModuleList of Head
      self.proj:  nn.Linear(num_heads * head_size, n_embd)
    forward(x) -> (B, T, n_embd)
    """

    def __init__(self, n_embd, num_heads, head_size, block_size, dropout=0.0):
        super().__init__()
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError


class FeedForward(nn.Module):
    """Per-token MLP: n_embd -> 4*n_embd -> ReLU -> n_embd. Both Linears have a bias.

    forward(x) -> same shape as x. Position t's output depends only on position t's input.
    """

    def __init__(self, n_embd, dropout=0.0):
        super().__init__()
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError

In [ ]:
grade(BigramLanguageModel=BigramLanguageModel,
      agg_loop=agg_loop, agg_tril=agg_tril, agg_softmax=agg_softmax,
      Head=Head, MultiHeadAttention=MultiHeadAttention, FeedForward=FeedForward, upto=4)

## Milestone 5 — the Block, and the whole GPT

A Block is communication then computation, and it has to be stackable: deep stacks need a
straight path for the gradient and a normalization that does not depend on the batch. Then
the model: the bigram's table gives a token *identity*; attention is a set operation, so the
model needs a second table that says *where* a token is.

`generate` is reused from the bigram, unchanged. That is why it crops to `block_size`.

**Block(n_embd, n_head, block_size, dropout=0.0)**
- Holds: `head_size = n_embd // n_head`; `self.sa = MultiHeadAttention(n_embd, n_head, head_size, block_size, dropout)`;
  `self.ffwd = FeedForward(n_embd, dropout)`; `self.ln1`, `self.ln2`, each `nn.LayerNorm(n_embd)`.
- `forward(x)`: `y = x + sa(ln1(x))`, then `y + ffwd(ln2(y))`.
- Returns the same shape as `x`. Causal.

**GPTLanguageModel(vocab_size, n_embd, block_size, n_head, n_layer, dropout=0.0)**
- Holds: `self.block_size`; a token table `nn.Embedding(vocab_size, n_embd)`; a position table
  `nn.Embedding(block_size, n_embd)`; `n_layer` Blocks; a final `nn.LayerNorm(n_embd)`;
  `lm_head = nn.Linear(n_embd, vocab_size)`.
- `forward(idx, targets=None)` for `idx` of shape `(B, T)`, `T <= block_size`:
  `logits = lm_head(ln_f(blocks(tok_emb(idx) + pos_emb(arange(T)))))`, shape `(B, T, vocab_size)`;
  `loss` is the bigram's contract (mean cross entropy over every `(b, t)`), `None` when
  `targets` is None.
- Returns `(logits, loss)`. The same token at two positions gives different logits. Logits at
  position `t` depend only on `idx[:, :t+1]`. Every parameter receives gradient. Initial loss
  is near `ln(vocab_size)`.

The grader checks, in order: Block has `sa` / `ffwd` / `ln1` / `ln2` → Block output shape →
values equal the two pre-norm residual adds (it says "no residual" or "post-norm" if your
output matches one of those) → Block is causal → GPT logits shape and `loss is None` →
loss equals cross entropy of its own logits → initial loss near `ln(vocab)` → `n_layer`
Blocks present → position matters → causal in token space → `T < block_size` and `generate`
→ every parameter gets gradient.

In [ ]:
class Block(nn.Module):
    """One transformer block: communication, then computation, each with a residual connection
    and a LayerNorm applied to the sublayer's INPUT (pre-norm).

    __init__(n_embd, n_head, block_size, dropout=0.0)
      head_size = n_embd // n_head
      self.sa   : MultiHeadAttention
      self.ffwd : FeedForward
      self.ln1, self.ln2 : nn.LayerNorm(n_embd)
    forward(x) -> same shape as x
    """

    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError


class GPTLanguageModel(nn.Module):
    """Token embedding + position embedding -> n_layer Blocks -> final LayerNorm -> lm_head.

    __init__(vocab_size, n_embd, block_size, n_head, n_layer, dropout=0.0)
      self.block_size must be set (generate reads it).
      Position embedding table has block_size rows.
    forward(idx, targets=None) -> (logits, loss), same contract as the bigram.
      idx: (B, T) with T <= block_size.
    """

    def __init__(self, vocab_size, n_embd, block_size, n_head, n_layer, dropout=0.0):
        super().__init__()
        self.block_size = block_size
        raise NotImplementedError

    def forward(self, idx, targets=None):
        raise NotImplementedError

    generate = BigramLanguageModel.generate

In [ ]:
grade(BigramLanguageModel=BigramLanguageModel,
      agg_loop=agg_loop, agg_tril=agg_tril, agg_softmax=agg_softmax,
      Head=Head, MultiHeadAttention=MultiHeadAttention, FeedForward=FeedForward,
      Block=Block, GPTLanguageModel=GPTLanguageModel, upto=5)

## Milestone 6 — it learns

The grader trains a tiny GPT (n_embd 32, 2 layers, 4 heads, block 32) for 600 steps on CPU,
about 10 seconds, and needs the val loss to beat what a bigram could ever reach. Predict the
number before you run it. Then train your own below and read the text.

In [ ]:
grade(BigramLanguageModel=BigramLanguageModel,
      agg_loop=agg_loop, agg_tril=agg_tril, agg_softmax=agg_softmax,
      Head=Head, MultiHeadAttention=MultiHeadAttention, FeedForward=FeedForward,
      Block=Block, GPTLanguageModel=GPTLanguageModel, upto=6)

In [ ]:
# once milestone 6 passes (~30 s): a slightly bigger run, then read what it writes
torch.manual_seed(1337)
model = GPTLanguageModel(vocab_size, n_embd=64, block_size=32, n_head=4, n_layer=2, dropout=0.0)
print(sum(p.numel() for p in model.parameters()) / 1e3, 'K parameters')
train(model, steps=1500, lr=1e-3, batch_size=32, block_size=32, eval_every=500)
print(decode(model.generate(torch.zeros((1, 1), dtype=torch.long), 400)[0].tolist()))

## Milestone 7 (stretch) — LayerNorm from scratch

You used `nn.LayerNorm` above. Write it. The only real question is *which axis* the statistics
are taken over, and why that makes it different from the batchnorm of lecture 4.

**LayerNorm1d(dim, eps=1e-5)**
- Holds: exactly two parameters, each of shape `(dim,)`: a per-channel scale (init ones) and
  shift (init zeros).
- `forward(x)` for `x` of shape `(..., dim)`: `scale * (x - mean) / sqrt(var + eps) + shift`,
  with `mean` and `var` (biased: divide by `dim`, not `dim - 1`) taken over the last axis only.
- Returns the same shape as `x`, matching `nn.LayerNorm(dim)` in value and in gradient, for
  both `(4, 8, dim)` and `(16, dim)` inputs.

The grader checks, in order: is an `nn.Module` → two `(dim,)` parameters → output shape →
values against `nn.LayerNorm` (it names the wrong axis or the unbiased variance if your output
matches one of those) → gradient matches → scale and shift are actually applied.

Nothing after this milestone depends on `LayerNorm1d`, so the final grade cell below passes
`skip=(7,)`. Remove the skip once you've written it. Then, if you want: swap it into `Block`
and `GPTLanguageModel` and confirm milestone 6 still passes. And think about dropout: the
stubs above take a `dropout` argument. Where in Head, MultiHeadAttention and FeedForward
does it go?

In [ ]:
class LayerNorm1d(nn.Module):
    """Normalize each vector along its LAST axis to mean 0 / var 1, then apply a learned
    per-channel scale and shift.

    __init__(dim, eps=1e-5): two parameters of shape (dim,).
    forward(x) -> same shape as x, for x of shape (..., dim). Must match nn.LayerNorm(dim).
    """

    def __init__(self, dim, eps=1e-5):
        super().__init__()
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError

In [ ]:
grade(BigramLanguageModel=BigramLanguageModel,
      agg_loop=agg_loop, agg_tril=agg_tril, agg_softmax=agg_softmax,
      Head=Head, MultiHeadAttention=MultiHeadAttention, FeedForward=FeedForward,
      Block=Block, GPTLanguageModel=GPTLanguageModel, LayerNorm1d=LayerNorm1d,
      skip=(7,))   # stretch; nothing later needs LayerNorm1d. Remove once written.


## Your own training loop and sampler

The notebook gave you `train` and `sample_from_logits`. Close them off (don't scroll up) and
write both from memory: build a small GPT, run an AdamW loop with periodic train/val loss, then
sample 300 characters from it with your own softmax-and-multinomial loop. If the loss does
something weird, ask.

In [ ]:
torch.manual_seed(0)
model = GPTLanguageModel(vocab_size, n_embd=32, block_size=32, n_head=4, n_layer=2)

for step in range(600):
    ...


## Scratch

Space to poke at things. Try printing the softmaxed `wei` of a trained head on a real batch and
look at which past characters each position attends to.